# Week 5 & 6 Deliverables   

## 1. Download Data

### Samples
- [Short-read Illumina (**interleaved** paired-end FASTQ)](https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/illumina.fq.bz2)
- [Long-read PacBio](https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/pacbio.fq.bz2)

### Reference Genome
The hg38 (or GRCh38) version of the human genome, focusing on the chromosome that contains these genes ([chromosome 10](https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr10.fa.gz)): 
- CYP2C8 (regulates many drugs, including anticancer, diabetes and blood pressure drugs)
- CYP2C9 (regulates many common drugs, including warfarin / Coumadin and NSAIDs such as Advil)
- CYP2C19 (regulates… yup, many common drugs, including antiplatelet drugs, antidepressants and anti-epileptic drugs).            

|Genes | CYP2C8 | CYP2C9 | CYP2C19 |
| --- | --- | --- | ---|
| Genomic sequence | chr10:95036772-95069497 | chr10:94938658-94990091 | chr10:94762681-94855547 |
| Strand | - | + | + | 
| Genomic size | 32726 | 51434 | 92867 |

In [ ]:
!mkdir -p data

# SAMPLES
# Download Illumina and PacBio data
!wget -P data/ https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/illumina.fq.bz2
!wget -P data/ https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/pacbio.fq.bz2
!bunzip2 data/*.bz2

# REFERENCE GENOME 
# chr10 containing CYP2C genes
!wget -P data/ https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr10.fa.gz
!gunzip data/*.gz

In [ ]:
# Locate the CYP2C8, CYP2C9, and CYP2C19 genes in the reference genome
# only the necessary
import pysam

path = "data/chr10.fa"
fasta = pysam.FastaFile(path)

GENE_INFO = {
    "CYP2C19": {"chr": "chr10", "start": 94762681, "end": 94855547, "strand": "+"},
    "CYP2C9":  {"chr": "chr10", "start": 94938658, "end": 94990091, "strand": "+"},
    "CYP2C8":  {"chr": "chr10", "start": 95036772, "end": 95069497, "strand": "-"},
}

with open("data/CYP2C.fa", "w") as out_f:
    for name, info in GENE_INFO.items():
        seq = fasta.fetch(info["chr"], info["start"] - 1, info["end"])
        out_f.write(f">{name} {info['chr']}:{info['start']}-{info['end']} ({info['strand']})\n")
        out_f.write(seq + "\n")

fasta.close()

In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import requests

# Output FASTA file
output_file = "data/reference_genome.fa"

# Gene coordinates (hg38, UCSC)
GENE_INFO = {
    "CYP2C19": {"chr": "chr10", "start": 94762681, "end": 94855547, "strand": "+"},
    "CYP2C9":  {"chr": "chr10", "start": 94938658, "end": 94990091, "strand": "+"},
    "CYP2C8":  {"chr": "chr10", "start": 95036772, "end": 95069497, "strand": "-"},
}

# UCSC FASTA API template
ucsc_fasta_url = "https://api.genome.ucsc.edu/getData/sequence?genome=hg38;chrom={chr};start={start};end={end}"

records = []

for gene, info in GENE_INFO.items():
    url = ucsc_fasta_url.format(chr=info["chr"], start=info["start"]-1, end=info["end"])
    r = requests.get(url)
    r.raise_for_status()
    seq = r.json()["dna"]

    # Create SeqRecord
    record = SeqRecord(
        Seq(seq),
        id=gene,
        description=f"{info['chr']}:{info['start']}-{info['end']} ({info['strand']})"
    )
    records.append(record)

# Write all genes to a single FASTA
with open(output_file, "w") as f:
    SeqIO.write(records, f, "fasta")

## 2. Align Samples to Reference Genome

**Short-read Illumina (interleaved paired-end FASTQ)**  
-x: applies multiple options at the same time 
sr: short read alignment without slicing

**Long-read PacBio**  
map-hifi: align PacBio high-fidelity reads to a reference genome

In [ ]:
# minimap index
!minimap2 -d data/reference_genome.mmi data/reference_genome.fa 

# Short read Illumina 
!minimap2 -ax sr data/reference_genome.mmi data/illumina.fq > data/illumina.sam 

# Long read BioPac
!minimap2 -ax map-hifi data/reference_genome.mmi data/pacbio.fq > data/pacbio.sam

In [ ]:
# Convert SAM to sorted BAM
!samtools view -bS data/illumina.sam | samtools sort -o data/illumina.sorted.bam
!samtools view -bS data/pacbio.sam | samtools sort -o data/pacbio.sorted.bam

# Index BAM for random access
!samtools index data/illumina.sorted.bam
!samtools index data/pacbio.sorted.bam

## 3. Variant Calling
- Find all variants (i.e., call them) in each sample for all genes of interest, and obtain the resulting VCF file. You can use either bcftools or FreeBayes. Those with suppressed masochistic tendencies are welcome to use GATK, but be warned: that tool is the epitome of (needless) complexity.

bcftools mpileup -f reference.fa alignments.bam | 
    mpileup part generates genotype likelihoods at each genomic position with coverage

bcftools call -mv -Ob -o calls.bcf
    call part makes the actual calls
    -m switch tells the program to use the default calling method
    -v option asks to output only variant sites
    -O option selects the output format

bcftools mpileup -Ou -f reference.fa alignments.bam | bcftools call -mv -Ob -o calls.bcf
    Do not waste computer’s time by making mpileup convert from the internal binary representation (BCF) to text (VCF), only to be immediately converted back to binary representation by call. Instead, use -Ou to work with uncompressed BCF output

- Expected output: two VCF files (one for each sample).

In [ ]:
# Index reference genome
pysam.faidx("data/reference_genome.fa")

In [ ]:
# Call variants 
!bcftools mpileup -Ou -f data/reference_genome.fa data/illumina.sorted.bam | bcftools call -mv --ploidy 2 -Oz -o data/illumina.vcf.gz
!bcftools index data/illumina.vcf.gz

!bcftools mpileup -Ou -f data/reference_genome.fa data/pacbio.sorted.bam | bcftools call -mv --ploidy 2 -Oz -o data/pacbio.vcf.gz
!bcftools index data/pacbio.vcf.gz

!gunzip data/*.gz

In [22]:
!freebayes -f data/reference_genome.fa --ploidy 2 data/illumina.sorted.bam >data/illumina_fb.vcf
!freebayes -f data/reference_genome.fa --ploidy 2 data/pacbio.sorted.bam >data/pacbio_fb.vcf

## 4. Phase Variant VCFs 
- phase the variant VCFs with HapCUT2 or HapTree-X. The output of these tools may be in HapCUT block format; if that happens, convert this file to the phased VCF format.

- Expected output: two VCF files (one for each sample).

In [ ]:
!extractHAIRS --bam data/illumina.sorted.bam --VCF data/illumina.vcf --out data/illumina.fragments --ref data/reference_genome.fa 
!HAPCUT2 --fragments data/illumina.fragments --VCF data/illumina.vcf --output data/illumina.hapcut

!extractHAIRS --pacbio 1 --bam data/pacbio.sorted.bam --VCF data/pacbio.vcf --out data/pacbio.fragments --ref data/reference_genome.fa 
!HAPCUT2 --fragments data/pacbio.fragments --VCF data/pacbio.vcf --output data/pacbio.hapcut

In [ ]:
!extractHAIRS --bam data/illumina.sorted.bam --VCF data/illumina_fb.vcf --out data/illumina_fb.fragments --ref data/reference_genome.fa 
!HAPCUT2 --fragments data/illumina_fb.fragments --VCF data/illumina_fb.vcf --output data/illumina_fb.hapcut

!extractHAIRS --pacbio 1 --bam data/pacbio.sorted.bam --VCF data/pacbio_fb.vcf --out data/pacbio_fb.fragments --ref data/reference_genome.fa 
!HAPCUT2 --fragments data/pacbio_fb.fragments --VCF data/pacbio_fb.vcf --output data/pacbio_fb.hapcut


Extracting haplotype informative reads from bamfiles data/illumina.sorted.bam minQV 13 minMQ 20 maxIS 1000 

VCF file data/illumina_fb.vcf has 7266 variants 
adding chrom CYP2C19 to index 

ERROR: Non-diploid VCF entry detected. Each VCF entry must have a diploid genotype (GT) field consisting of two alleles in the set {0,1,2} separated by either '/' or '|'. For example, "1/1", "0/1", and "0|2" are valid diploid genotypes for HapCUT2, but "1", "0/3", and "0/0/1" are not.
The invalid entry is: 

CYP2C19	10343	.	CACCCG	TGCCCA,TGCCCG,CGCCCG,CACCTG,CACCCA	7.95535e-10	.	AB=0.0521739,0.0556522,0.135652,0.126957,0.158261;ABP=1004.63,989.127,666.012,698.036,586.283;AC=0,0,0,0,1;AF=0,0,0,0,0.5;AN=2;AO=30,32,78,73,91;CIGAR=2X3M1X,2X4M,1M1X4M,4M1X1M,5M1X;DP=575;DPB=576.167;DPRA=0,0,0,0,0;EPP=3.0103,4.09604,3.0103,3.04005,5.89764;EPPR=5.57421;GTI=0;LEN=6,2,1,1,1;MEANALT=21,21,21,21,21;MQM=7.4,13.0625,6.78205,7.21918,10.9121;MQMR=25.8253;NS=1;NUMALT=5;ODDS=22.4209;PAIRED=0.0333333,0.15625,0.179487

## 5. Variant Analysis
- Now you should have two phased VCF files (one for each sequencing technology). Compare these VCFs. 
    - How many variants are shared between the VCFs? How many are not?
- Select 2-3 variants that are not common (if any) and check which technology supports this variant. Open both BAM files in IGV and take a screenshot of each problematic discordant location. What can you deduce from these screenshots—are these variants sequencing-related artifacts or are they indeed true variants? Do this analysis for every gene.
- IGV screenshots can also be automated (it is a bit tricky, though—ask your LLM for help). You can opt out of doing this, but you will lose half a point.

- Expected output: Jupyter cell(s) with IGV screenshots and a discussion.

## 6. Star-Allele Calls
- Can you figure out the star-allele for each gene of interest? The star-allele database can be found in PharmVar; see this for CYP2C19. Your answer should be something like CYP2C19*12 because X, Y and Z. This step does not have to be automated, but should be at least explained in the notebook.
    - Hint: use phased data!

- Expected output: Jupyter cell(s) with discussion (and code, if you want to do it that way).